In [ ]:
import os

# Working directory must contain AlphaSimPy.py for imports
os.chdir(r"/Users/mtwatson/Library/CloudStorage/Box-Box/Projects/AI agent for breeding/Endpoint 2 agent")


# Peixoto2 AlphaSimPy Notebook

This notebook converts the provided BRAID breeding program abstraction into a tutorial-style **AlphaSimPy** simulation workflow.

**Program summary**
- Doubled haploid breeding program
- Biparental crossing from 50 founders
- DH derivation followed by three testcross-style evaluation stages: TC1, TC2, and TC3
- Final release of 2 varieties
- One additive trait: `overall_merit`

**BRAID-derived horizon**: 4 years  
**Package**: AlphaSimPy


## Assumptions and translation notes

The BRAID abstraction specifies stage structure, sizes, and evaluation intensity, but some AlphaSimPy implementation details are not fully defined. This notebook uses the following explicit assumptions:

1. **Founders** are simulated with `runMacs` because no external genotype file was provided.
2. **One additive trait** is simulated with 100 QTL total and heritability 0.3.
3. **DH production** is represented with `makeDH` from F1 individuals.
4. **Testcross evaluation with GCA/genomic prediction** is approximated as staged phenotypic evaluation with increasing effective accuracy across TC1, TC2, and TC3.
5. **Selection** is implemented with `selectInd` on phenotypes to preserve the BRAID stage sizes.
6. The notebook tracks **genetic mean** and **genetic variance** at each stage. Inbreeding is noted in the BRAID outputs, but this notebook focuses on metrics directly available from standard AlphaSimPy tutorial functions.

These assumptions are documented so the notebook remains readable and executable while staying faithful to the BRAID design.

## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from AlphaSimPy import (
    runMacs,
    SimParam,
    newPop,
    randCross,
    makeDH,
    setPheno,
    selectInd,
    meanG,
    varG,
)

print('AlphaSimPy notebook for Peixoto2 loaded successfully')


## Global Parameters

These values are taken directly from the BRAID abstraction where possible.

In [ ]:
# Program-level settings
program_name = 'Peixoto2'
n_years = 4
np.random.seed(12345)

# Genome and trait settings from BRAID
n_chr = 10
founder_size = 50
n_qtl_total = 100
h2 = 0.3
error_variance = 1.0

# Stage sizes from BRAID
n_crosses = 50
parents_per_cross = 2
n_f1 = 50
n_dh = 1250
n_tc1 = 1250
n_tc1_selected = 800
n_tc2_selected = 200
n_tc3_selected = 100
n_release = 2

# Evaluation settings from BRAID
tc1_env = 2
tc2_env = 5
tc3_env = 20
replications = 1

print(program_name)
print(f'Founders: {founder_size}, Crosses: {n_crosses}, DH lines: {n_dh}')


## Helper Functions

A small helper is used to record stage summaries in a tidy table.

In [ ]:
records = []

def recordStage(stage_name, pop, year):
    records.append({
        'year': year,
        'stage': stage_name,
        'n_ind': pop.nInd,
        'meanG': float(meanG(pop)[0]) if np.ndim(meanG(pop)) > 0 else float(meanG(pop)),
        'varG': float(varG(pop)[0]) if np.ndim(varG(pop)) > 0 else float(varG(pop)),
    })


## Simulate Founders and Create the Base Population

The BRAID file indicates 50 external founders. Since no genotype file was supplied, we simulate founders with `runMacs`.

In [ ]:
# Simulate founder haplotypes
founder_haplo = runMacs(nInd=founder_size, nChr=n_chr, segSites=200)

# Set simulation parameters
SP = SimParam(founder_haplo)
SP.addTraitA(nQtlPerChr=max(1, n_qtl_total // n_chr), mean=0.0, var=1.0)
SP.setVarE(h2=h2)

# Create founder population
founders = newPop(founder_haplo, simParam=SP)
recordStage('founders', founders, year=1)

print('Founder population created')
print('Mean genetic value:', meanG(founders))
print('Genetic variance:', varG(founders))


## Year 1: Biparental Crossing and DH Derivation

The BRAID workflow specifies 50 biparental crosses followed by DH derivation.

In [ ]:
# Create F1 individuals from biparental crosses
f1 = randCross(founders, nCrosses=n_crosses, nProgeny=1, simParam=SP)
recordStage('f1', f1, year=1)

# Derive doubled haploid lines from F1
dh_per_f1 = int(n_dh / n_f1)
dh = makeDH(f1, nDH=dh_per_f1, simParam=SP)
recordStage('dh', dh, year=1)

print('F1 size:', f1.nInd)
print('DH size:', dh.nInd)


## Year 2: TC1 Candidate Preparation, Evaluation, and Selection

TC1 is represented as the first evaluation stage with 2 environments and selection of 800 lines.

In [ ]:
# Prepare TC1 candidates
tc1_candidates = dh
recordStage('tc1_candidates', tc1_candidates, year=2)

# Evaluate TC1 candidates
tc1_candidates = setPheno(tc1_candidates, varE=error_variance / tc1_env, simParam=SP)

# Select top individuals after TC1
tc1_selected = selectInd(tc1_candidates, nInd=n_tc1_selected, use='pheno', simParam=SP)
recordStage('tc1_selected', tc1_selected, year=2)

print('TC1 candidates:', tc1_candidates.nInd)
print('TC1 selected:', tc1_selected.nInd)


## Year 3: TC2 Evaluation and Selection

TC2 uses more environments than TC1, so the effective error variance per entry mean is reduced.

In [ ]:
# Evaluate TC2-selected candidates
tc1_selected = setPheno(tc1_selected, varE=error_variance / tc2_env, simParam=SP)
tc2_selected = selectInd(tc1_selected, nInd=n_tc2_selected, use='pheno', simParam=SP)
recordStage('tc2_selected', tc2_selected, year=3)

print('TC2 selected:', tc2_selected.nInd)


## Year 4: TC3 Evaluation, Final Selection, and Variety Release

The final stage evaluates 200 lines in 20 environments, selects 100, and releases the top 2 varieties.

In [ ]:
# Evaluate TC3 candidates
tc2_selected = setPheno(tc2_selected, varE=error_variance / tc3_env, simParam=SP)
tc3_selected = selectInd(tc2_selected, nInd=n_tc3_selected, use='pheno', simParam=SP)
recordStage('tc3_selected', tc3_selected, year=4)

# Final release selection
released_varieties = setPheno(tc3_selected, varE=error_variance / tc3_env, simParam=SP)
released_varieties = selectInd(released_varieties, nInd=n_release, use='pheno', simParam=SP)
recordStage('released_varieties', released_varieties, year=4)

print('TC3 selected:', tc3_selected.nInd)
print('Released varieties:', released_varieties.nInd)


## Results Summary

In [ ]:
results = pd.DataFrame(records)
results


## Plot Genetic Mean and Variance Across Stages

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

axes[0].plot(range(len(results)), results['meanG'], marker='o')
axes[0].set_ylabel('Mean Genetic Value')
axes[0].set_title('Genetic Mean Across Program Stages')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(len(results)), results['varG'], marker='o', color='darkorange')
axes[1].set_ylabel('Genetic Variance')
axes[1].set_title('Genetic Variance Across Program Stages')
axes[1].set_xticks(range(len(results)))
axes[1].set_xticklabels(results['stage'], rotation=45, ha='right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Conclusion

This notebook translates the BRAID abstraction for **Peixoto2** into an executable AlphaSimPy workflow. It preserves the main program structure:

- 50 founders
- 50 biparental crosses
- 1,250 DH lines
- TC1 -> TC2 -> TC3 stage progression
- Final release of 2 varieties

Where the BRAID abstraction did not fully specify implementation details for genomic prediction and testcross structure, transparent assumptions were used and documented.